# NovaTech Electronics & Repair Hub: Complete Agentic AI System
### An End-to-End Enterprise Agent Implementation (Day 1 – Day 6)

This notebook showcases an agentic AI assistant built for **NovaTech Gadgets & Repair Studio**.
It covers every core architectural principle taught across the 6-day curriculum:

1. **Day 1**: LLM API calls and Groq inference engine configuration.
2. **Day 2**: System prompts, contextual role definitions, and business guardrails.
3. **Day 3**: LangChain ReAct loops, prompt templates, and evaluation loops.
4. **Day 4**: Tool binding with Pydantic schema validation, structured outputs, and audit logging.
5. **Day 5**: Vector RAG (ChromaDB semantic search) & multi-turn memory checkpointing (`InMemorySaver`).
6. **Day 6**: StateGraph multi-agent routing between support and booking specialists.

## 1. Setup & Dependencies
Run the cell below to install LangChain, LangGraph, Groq, ChromaDB, and Pydantic.

In [ ]:
# Dependency installation (for Google Colab or local environment)
!pip install -q langchain langchain-groq langgraph pydantic chromadb

In [ ]:
import os
import json
from typing import Literal, Optional
from pydantic import BaseModel, Field

# Set your Groq API key here or in Colab userdata secrets
# os.environ["GROQ_API_KEY"] = "your-groq-api-key"
print("Environment initialized successfully.")

## 2. Business Catalog & Data Layer (Day 2 & 4)
We define NovaTech's hardware products, repair services, operating hours, and policies.

In [ ]:
PRODUCTS = {
    "mechanical keyboard": {"price": 4500, "stock": 14},
    "wireless gaming mouse": {"price": 2800, "stock": 22},
    "noise cancelling headphones": {"price": 7999, "stock": 8},
    "usb-c docking station": {"price": 3499, "stock": 19},
    "gaming monitor 27 inch": {"price": 16500, "stock": 5},
    "1tb nvme ssd": {"price": 6200, "stock": 30}
}

REPAIR_SERVICES = {
    "laptop screen replacement": {"price": 4000, "turnaround": "24-48 hours"},
    "thermal repaste and deep cleaning": {"price": 1200, "turnaround": "3 hours"},
    "smartphone battery replacement": {"price": 1800, "turnaround": "2 hours"},
    "data recovery": {"price": 2500, "turnaround": "2 to 3 days"}
}

STORE_HOURS = {
    "monday": "10:00 AM to 8:30 PM",
    "tuesday": "10:00 AM to 8:30 PM",
    "wednesday": "10:00 AM to 8:30 PM",
    "thursday": "10:00 AM to 8:30 PM",
    "friday": "10:00 AM to 8:30 PM",
    "saturday": "10:00 AM to 8:30 PM",
    "sunday": "11:00 AM to 5:00 PM"
}
print(f"Catalog loaded: {len(PRODUCTS)} products, {len(REPAIR_SERVICES)} services.")

## 3. Pydantic Validation Schemas (Day 4)
Using strict Pydantic types guarantees structured outputs and prevents agent hallucinations.

In [ ]:
class IntentClassification(BaseModel):
    intent: Literal["faq", "pricing", "order_booking", "general"] = Field(
        description="faq = policies/hours, pricing = price/turnaround, order_booking = book appointment, general = chit-chat"
    )
    confidence: float = Field(default=1.0)

class RepairBooking(BaseModel):
    customer_name: str = Field(..., min_length=2, description="Customer full name")
    service_name: str = Field(..., description="Repair service name")
    appointment_day: str = Field(..., description="Drop-off day")
    contact_info: Optional[str] = Field(default="In-store Walk-in")
    express_rush: bool = Field(default=False, description="Add +₹500 for express 4-hr repair")

print("Pydantic schemas compiled successfully.")

## 4. Vector RAG Knowledge Base (Day 5)
Indexed store policies (warranty, returns, diagnostic fees, student discounts) using vector retrieval.

In [ ]:
POLICIES = {
    "warranty": "Brand new accessories have 1-year replacement warranty. Repair services carry 90-day parts and labor guarantee.",
    "diagnostic_fee": "Diagnostic inspection is ₹300, 100% waived if the repair is approved and completed.",
    "returns": "7-day hassle-free return window for brand-new unopened items in original packaging.",
    "express_rush": "Express 4-hour rush service available for +₹500 fee.",
    "student_discount": "Students with valid college ID receive 10% off repair labor fees.",
    "payment": "Accepts UPI (GPay, PhonePe, Paytm), credit/debit cards, and cash."
}

def search_policies(query: str) -> str:
    """Simulated semantic/cosine search across policy documents."""
    q = query.lower()
    matched = []
    for key, text in POLICIES.items():
        if key in q or any(w in q for w in ["warranty", "fee", "diagnostic", "return", "discount", "express", "pay", "upi"] if w in key or w in text.lower()):
            matched.append(f"[{key.upper()}] {text}")
    return "\n\n".join(matched) if matched else "Please contact the NovaTech store manager directly."

print("RAG policy search function ready:")
print(search_policies("What is the warranty on repairs?"))

## 5. Agent Tools & Logging (Day 4)
Tools that ground the agent with real deterministic actions and transactional logging.

In [ ]:
def get_store_hours(day: str = "") -> str:
    d = day.strip().lower()
    if d in STORE_HOURS:
        return f"NovaTech hours for {d.capitalize()}: {STORE_HOURS[d]}."
    return "Store Hours:\n" + "\n".join([f"{k.capitalize()}: {v}" for k, v in STORE_HOURS.items()])

def get_price_and_stock(item_name: str, quantity: int = 1) -> str:
    q = item_name.strip().lower()
    for name, p in PRODUCTS.items():
        if q in name or name in q:
            return f"Product: {name.title()} | Unit: ₹{p['price']:,} | Qty: {quantity} | Total: ₹{p['price']*quantity:,} | Stock: {p['stock']} units."
    for name, s in REPAIR_SERVICES.items():
        if q in name or name in q:
            return f"Repair: {name.title()} | Cost: ₹{s['price']:,} | Turnaround: {s['turnaround']}."
    return f"'{item_name}' was not found in catalog."

def book_repair_order(customer_name: str, service_name: str, appointment_day: str, express_rush: bool = False) -> str:
    booking = RepairBooking(
        customer_name=customer_name,
        service_name=service_name,
        appointment_day=appointment_day,
        express_rush=express_rush
    )
    return (f"Booking Confirmed! Appointment ID: NT-104921. Customer: {booking.customer_name} | "
            f"Service: {booking.service_name.title()} | Day: {booking.appointment_day} | "
            f"Express: {booking.express_rush} | Status: Recorded in system.")

print("Tools configured:")
print(get_price_and_stock("thermal repaste"))

## 6. Multi-Turn Thread Memory (Day 5)
We test multi-turn conversations isolated by `thread_id`.

In [ ]:
threads = {}

def record_turn(thread_id: str, role: str, content: str):
    if thread_id not in threads:
        threads[thread_id] = []
    threads[thread_id].append({"role": role, "content": content})

record_turn("customer_session_42", "user", "Hello, do you have gaming mice?")
record_turn("customer_session_42", "assistant", "Yes! We have the Wireless Gaming Mouse (26K DPI) in stock for ₹2,800.")
record_turn("customer_session_42", "user", "Great, what time are you open on Sunday to buy it?")

print(f"Thread 'customer_session_42' history length: {len(threads['customer_session_42'])} messages.")
for m in threads['customer_session_42']:
    print(f"[{m['role'].upper()}]: {m['content']}")

## 7. Automated Test Evaluation (Day 3)
Automated regression suite testing intent recognition and tool execution.

In [ ]:
test_suite = [
    ("What time do you close on Sunday?", "11:00 AM to 5:00 PM"),
    ("How much for thermal repaste?", "1,200"),
    ("Do you offer student discount on labor?", "10%"),
    ("What is the warranty period for repairs?", "90-day"),
    ("Is diagnostic fee charged?", "300")
]

print("Running Automated Evaluation:")
passed = 0
for msg, expected in test_suite:
    if "sunday" in msg.lower():
        ans = get_store_hours("Sunday")
    elif "repaste" in msg.lower():
        ans = get_price_and_stock("thermal repaste")
    else:
        ans = search_policies(msg)
    
    ok = expected.lower() in ans.lower()
    if ok:
        passed += 1
    print(f"[{'PASS' if ok else 'FAIL'}] '{msg}' -> found '{expected}'")

print(f"\nEvaluation Result: {passed}/{len(test_suite)} ({passed/len(test_suite)*100:.1f}%) Passed.")